This fixture-only notebook runs the governed systems-optimization command: development search first, published post-search selection second, and final comparison last. The workloads are published literals, and the boundary is that the two child processes holding candidate authority -- the one that runs a candidate scheduler and the one that decides what to propose -- cannot read or write this repository's files through the tested file route, so neither can open the later splits. The trusted driver and evaluator own those splits and read them by design. There is no supplied-proposer seam. This is a mechanism example, not a reproduction of ADRS; the limits cell below says what the boundary does not cover.

In [1]:
from importlib import import_module
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

command = import_module("examples.research.systems_optimization.command")

# The directory only has to outlive the run: every cell below reads the
# retained decision and replay values this one keeps in memory. A context
# manager therefore removes it whether the command returns or raises, where a
# cleanup cell at the end is skipped by any failure before it.
with TemporaryDirectory(prefix="meta-evolve-systems-") as name:
    output = Path(name)
    result = command.run_example(output)

fixture_only = result.decision.proposer == command.FIXTURE_PROPOSER.label
print("fixture-only run:", fixture_only)
print("decision digest:", result.decision_digest)
print("temporary output cleaned:", not output.exists())

fixture-only run: True
decision digest: 6cd280071800a1b0
temporary output cleaned: True


## Development search

The deterministic fixture proposes from development feedback only. Every attempted candidate remains visible, including typed refusals that have no score.

In [2]:
for attempt in result.decision.development:
    measured = (
        f"quality={attempt.quality:.4f} work={attempt.work:.1f}"
        if attempt.rankable
        else f"failure={attempt.failure_kind}"
    )
    print(
        f"{attempt.logical_step}: {attempt.candidate}: "
        f"{attempt.outcome} {measured}"
    )

0: round-robin (seed): completed quality=0.7929 work=0.0
1: most-loaded-of-two: completed quality=0.7731 work=2.0
2: least-loaded-scan: completed quality=0.8219 work=7.0
3: least-loaded-min-key: completed quality=0.8219 work=7.0
4: power-of-two-choices: completed quality=0.9136 work=2.0
5: places-off-the-cluster: invalid-schedule failure=invalid_output
6: refactor-into-a-helper: off-surface failure=policy_violation
7: scan-past-the-budget: constraint-violated failure=policy_violation
8: unparseable-rewrite: malformed-source failure=invalid_output


## Published selection

After the search returns, every development candidate that ranked is evaluated on the published selection split. The retained policy orders higher quality first, then lower work, then earlier logical step.

In [3]:
for position, attempt in enumerate(result.decision.selection.ranked, start=1):
    print(
        f"{position}: {attempt.candidate}: "
        f"quality={attempt.quality:.4f} work={attempt.work:.1f}"
    )
print("selected:", result.decision.selection.winner)

1: power-of-two-choices: quality=0.9014 work=2.0
2: least-loaded-scan: quality=0.8275 work=6.7
3: least-loaded-min-key: quality=0.8275 work=6.7
4: round-robin (seed): quality=0.7536 work=0.0
5: most-loaded-of-two: quality=0.6070 work=2.0
selected: power-of-two-choices


## Final comparison

Selection fixes the winner before the final split runs. Only the seed baseline and that selected candidate are evaluated here.

In [4]:
def reading(attempt):
    """A score when there is one, and the typed failure when there is not."""
    if attempt.rankable:
        return f"quality={attempt.quality:.4f}"
    return f"{attempt.outcome}: {attempt.failure_kind}"


comparison = result.decision.comparison
if comparison is None:
    print("no comparison: the search selected nothing to compare")
else:
    print(f"baseline: {comparison.baseline.candidate} {reading(comparison.baseline)}")
    print(f"selected: {comparison.selected.candidate} {reading(comparison.selected)}")
    print("verdict:", comparison.verdict)

baseline: round-robin (seed) quality=0.7912
selected: power-of-two-choices quality=0.8806
verdict: improved


## Performed work

These are counts reconstructed from retained attempts and scenario observations, not estimates and not merely the declared ceilings.

In [5]:
accounting = result.decision.accounting
print("evaluations:", dict(accounting.evaluations))
print("scenario runs:", dict(accounting.scenario_runs))
print("scenario ceiling:", accounting.scenario_ceiling)

evaluations: {'development': 9, 'private': 5, 'held_out': 2}
scenario runs: {'development': 13, 'private': 15, 'held_out': 4}
scenario ceiling: 49


## Replay from the directory alone

The replay reads the durable search store and the content-addressed record. It runs no simulator, candidate subprocess, or evaluator.

In [6]:
for name, agrees in result.replayed.items():
    print(f"{name}: {agrees}")
print("all replay checks:", all(result.replayed.values()))
assert all(result.replayed.values())

store holds the recorded run: True
run declaration agrees: True
every development attempt agrees: True
search best agrees: True
lineage agrees: True
evidence agrees: True
development accounting agrees: True
every attempt has exactly one timing: True
development timing agrees: True
total timing agrees: True
live decision matches: True
all replay checks: True


## What the directory does not prove

The durable store witnesses development work and its starting declaration. It did not run the two later evaluators, so a coherently rewritten later measurement, later evaluator declaration, or later clock reading has no independent local witness -- and neither does the retained scenario ceiling, which is a declared bound rather than a count. The record is tamper-evident, not immutable, and promotion remains external. Network, CPU, and memory are not confined either; the repository is, apart from the interpreter installation inside it, which the boundary refuses to exempt if that exemption would ever reach this example's own directory.

In [7]:
# The run directory was removed when its context manager closed, above.
print("temporary output cleaned:", not output.exists())

temporary output cleaned: True
